# Add `rating_band` to the preprocessed reviews

Reads `data/processed/digikala-comments-preprocessed.csv`, which already has `rate`. Adds the ordered band from the project definition and writes a new file. `rate` stays in the file. The band is the column a rating-band model should use.

- `low`: 0 ≤ rate < 2.5, encoded 0
- `medium`: 2.5 ≤ rate < 4, encoded 1
- `high`: 4 ≤ rate ≤ 5, encoded 2
- `invalid`: missing, non-numeric, or outside [0, 5]. The ordinal column is null for these rows.

Run this notebook from the repository root, or from this folder.


In [1]:
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "data").is_dir() and (ROOT.parent / "data").is_dir():
    ROOT = ROOT.parent

SOURCE = ROOT / "data/processed/digikala-comments-preprocessed.csv"
DEST = ROOT / "data/processed/digikala-comments-preprocessed-rating-band.parquet"
if not SOURCE.is_file():
    raise FileNotFoundError(SOURCE)
print(SOURCE)
print(DEST)


/Users/amir/Documents/GitHub/digikala-products-recommender-systems/data/processed/digikala-comments-preprocessed.csv
/Users/amir/Documents/GitHub/digikala-products-recommender-systems/data/processed/digikala-comments-preprocessed-rating-band.parquet


In [2]:
import polars as pl

frame = pl.read_csv(
    SOURCE,
    schema_overrides={
        "id": pl.Int64,
        "body": pl.String,
        "product_id": pl.Int64,
        "recommendation_status": pl.String,
        "rate": pl.Float64,
    },
)
rate = pl.col("rate")
band = (
    pl.when(rate.is_null()).then(pl.lit("invalid"))
    .when((rate >= 0) & (rate < 2.5)).then(pl.lit("low"))
    .when((rate >= 2.5) & (rate < 4)).then(pl.lit("medium"))
    .when((rate >= 4) & (rate <= 5)).then(pl.lit("high"))
    .otherwise(pl.lit("invalid"))
)
band_id = band.replace_strict({"low": 0, "medium": 1, "high": 2}, default=None)
labeled = frame.with_columns(
    band.alias("rating_band"),
    band_id.cast(pl.Int8).alias("rating_band_id"),
)
print(labeled.height, "rows")
print(labeled.group_by("rating_band").len().sort("rating_band"))
labeled.head(5)


5258342 rows
shape: (3, 2)
┌─────────────┬─────────┐
│ rating_band ┆ len     │
│ ---         ┆ ---     │
│ str         ┆ u32     │
╞═════════════╪═════════╡
│ high        ┆ 3320089 │
│ low         ┆ 990998  │
│ medium      ┆ 947255  │
└─────────────┴─────────┘


id,body,product_id,recommendation_status,rate,rating_band,rating_band_id
i64,str,i64,str,f64,str,i8
53672599,"""به درد نمیخوره""",252058,"""not_recommended""",1.0,"""low""",0
9897229,"""می‌تونس به عنو یه کالا فرهنگ ب…",252058,"""recommended""",0.0,"""low""",0
38074516,"""بسته بند خوب بود کاربرد و کیفی…",3331597,"""recommended""",0.0,"""low""",0
18628562,"""به نظر خوبه فقط یک ظریفه . از …",3331329,"""recommended""",0.0,"""low""",0
53301258,"""معمولیه اگه واسه خونه رنگ کردن…",3255700,"""recommended""",3.0,"""medium""",1


In [3]:
DEST.parent.mkdir(parents=True, exist_ok=True)
labeled.write_parquet(DEST)
print("saved", DEST, DEST.stat().st_size, "bytes")


saved /Users/amir/Documents/GitHub/digikala-products-recommender-systems/data/processed/digikala-comments-preprocessed-rating-band.parquet 150310825 bytes


In [4]:
labeled.write_csv(DEST.with_suffix(".csv"))

In [5]:
labeled.head()

id,body,product_id,recommendation_status,rate,rating_band,rating_band_id
i64,str,i64,str,f64,str,i8
53672599,"""به درد نمیخوره""",252058,"""not_recommended""",1.0,"""low""",0
9897229,"""می‌تونس به عنو یه کالا فرهنگ ب…",252058,"""recommended""",0.0,"""low""",0
38074516,"""بسته بند خوب بود کاربرد و کیفی…",3331597,"""recommended""",0.0,"""low""",0
18628562,"""به نظر خوبه فقط یک ظریفه . از …",3331329,"""recommended""",0.0,"""low""",0
53301258,"""معمولیه اگه واسه خونه رنگ کردن…",3255700,"""recommended""",3.0,"""medium""",1
